# Tokenizacija, rečnik i ID-jevi

Ova sveska vizuelno prikazuje kako tekst postaje lista tokena, pa niz celih brojeva. Funkcije su u `src/`; notebook ih samo poziva. Rečnik iz pravih podataka pravimo **isključivo iz trening skupa**.

## Mali primer koji možeš da proveriš ručno

U ovim izmišljenim porukama `free` i `now` se pojavljuju po dva puta. Ostali tokeni se pojavljuju po jednom. Rečnik sortira tokene prvo po učestalosti, a pri izjednačenju abecedno. Zato su ID-jevi ponovljivi, ali nisu mera značenja reči.

In [1]:
from collections import Counter
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.tokenization import tokenize_sms
from src.vocabulary import build_vocabulary, encode_sms

examples = ["Free prize now!", "Free entry now."]
toy_vocabulary = build_vocabulary(examples)
frequencies = Counter(token for message in examples for token in tokenize_sms(message))
pd.DataFrame(
    {"token": list(toy_vocabulary),
     "učestalost": [frequencies.get(token, 0) for token in toy_vocabulary],
     "ID": list(toy_vocabulary.values())}
)

,token,učestalost,ID
0,<PAD>,0,0
1,<UNK>,0,1
2,free,2,2
3,now,2,3
4,!,1,4
5,.,1,5
6,entry,1,6
7,prize,1,7


`<PAD>` uvek dobija ID `0` i kasnije će popunjavati kraće poruke. `<UNK>` dobija ID `1`: ako token nije viđen tokom pravljenja rečnika, koristimo ovaj ID. Nijedan od ta dva ID-ja nije nasumičan.

In [2]:
message = "Free offer!"
pd.DataFrame({"token": tokenize_sms(message), "ID": encode_sms(message, toy_vocabulary)})

,token,ID
0,free,2
1,offer,1
2,!,4


U prethodnoj tabeli `offer` ima ID `1`, jer ga nema u dve trening poruke iz primera. Rečnik se **ne proširuje** kada obrađujemo novu poruku.

## Rečnik iz pravog trening skupa

Ovde čitamo samo `sms_train.csv`. Izdvojeni `sms_test.csv` ne koristimo za građenje rečnika. Tokom kasnije cross-validacije isti postupak ćemo ponoviti na trening delu svakog fold-a.

In [3]:
train = pd.read_csv(project_root / "data" / "processed" / "sms_train.csv")
vocabulary = build_vocabulary(train["text"])
print("Trening poruka:", len(train))
print("Tokena u rečniku, zajedno sa <PAD> i <UNK>:", len(vocabulary))
print("Primer poruke:", train.loc[0, "text"])
tokens = tokenize_sms(train.loc[0, "text"])
ids = encode_sms(train.loc[0, "text"], vocabulary)
pd.DataFrame({"token": tokens, "ID": ids}).head(12)

Trening poruka: 4127
Tokena u rečniku, zajedno sa <PAD> i <UNK>: 7842
Primer poruke: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


,token,ID
0,go,63
1,until,578
2,jurong,5673
3,point,856
4,",",6
5,crazy,649
6,.,2
7,.,2
8,available,647
9,only,85


## Šta još nedostaje?

Sada imamo niz ID-jeva različite dužine za svaku poruku. Sledeći mali korak je da ograničimo dužinu, kraće nizove dopunimo ID-jem `0` i napravimo masku koja modelu kaže šta je prava poruka, a šta dopuna. Model još nije napravljen.